# 09. 영상 파이프라인 (좌표 -> 각도 -> 경기 집계)

> **핵심 로직은 `video_features.py`로 모듈화**, 이 노트북은 **import해서 호출만** 한다.
> (정형 쪽 `feature_aggregator.py`와 동일 패턴 -- 노트북=실행/기록, .py=재사용 함수)

| 단계 | 함수 (video_features.py) | 입력 -> 출력 |
|------|--------------------------|-------------|
| A. 좌표 합치기 | `merge_coords()` | `batch_slot*_coords.csv` -> 좌표+경기정보 |
| B. 각도 계산 | `compute_angles()` | 좌표 -> 공별 생체역학 각도 9종 (좌투 미러링·정규화) |
| C. 경기 집계 | `aggregate()` | 각도 -> 경기 단위 피처 (N구 x 집계) |
| **전체 일괄** | `build_all()` | 위 3단계 -> `features_bio_pitch{N}_{agg}.parquet` 일괄 생성 |

재실행: 영상이 더 추출되면 이 노트북만 Run All. 집계 결과는 `12_biomechanical_experiment`가 읽는다.

## 0. 환경·경로 설정

In [ ]:
import os, sys

# 환경 자동 감지 (Colab / 로컬)
IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    BASE       = '/content/drive/MyDrive/MLB_pitcher'
    MODULE_DIR = f'{BASE}/2_video'          # video_features.py 위치
    OUTPUT_DIR = f'{BASE}/output'           # batch_slot*_coords.csv
    PLAY_IDS   = f'{BASE}/data/play_ids_sample.csv'
    FEAT_DIR   = f'{BASE}/data/4_features'  # 출력 parquet
else:
    BASE       = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'
    MODULE_DIR = os.path.join(BASE, '2_video')
    OUTPUT_DIR = os.path.join(BASE, '0_data', 'output')
    PLAY_IDS   = os.path.join(BASE, '0_data', 'data', 'play_ids_sample.csv')
    FEAT_DIR   = os.path.join(BASE, '0_data', '4_features')

# video_features.py 를 import 경로에 추가
sys.path.insert(0, MODULE_DIR)
import video_features as vf
import importlib; importlib.reload(vf)   # 모듈 수정 시 반영

SLOTS = [0, 1, 2, 3, 4]   # 0,1,2=내 몫(2021~2023) + 3,4=조원(2024,2025)
print(f'환경: {"Colab" if IN_COLAB else "로컬"} | 모듈: {MODULE_DIR}')
print(f'좌표: {OUTPUT_DIR} | 슬롯: {SLOTS}')

## 1. 전체 일괄 실행 (`build_all`)

`merge_coords -> compute_angles -> aggregate`를 한 번에. N(3/5/10/15) x agg(mean/mean_std/full9) 조합 parquet을 모두 생성.

In [ ]:
angles = vf.build_all(
    output_dir=OUTPUT_DIR,
    play_ids_csv=PLAY_IDS,
    out_dir=FEAT_DIR,
    slots=SLOTS,
)
print(f'\n완료. all_angles.csv + features_bio_pitch*_*.parquet -> {FEAT_DIR}')
print(f'각도 {len(angles):,}행 | 시즌: {angles["season"].value_counts().sort_index().to_dict()}')

## 2. (선택) 단계별 실행 -- 중간 결과 확인용

`build_all` 대신 단계를 나눠 돌리며 중간 산출물을 보고 싶을 때.

In [ ]:
# A. 좌표 합치기
coords = vf.merge_coords(OUTPUT_DIR, PLAY_IDS, slots=SLOTS)
print(f'A. 좌표: {len(coords):,}행 | 경기 {coords.groupby(["game_pk","pitcher"]).ngroups:,}개')

# B. 각도 계산 (좌투 미러링 포함)
ang = vf.compute_angles(coords)
print(f'B. 각도: {len(ang):,}행')
print(ang[vf.ANGLE_COLS].describe().round(2).T[['mean','std','min','max']].to_string())

# C. 경기 집계 (예: pitch15 full9)
feat = vf.aggregate(ang, n_pitches=15, agg='full9')
print(f'\nC. 집계(pitch15/full9): 경기 {len(feat):,}개 · 피처 {feat.shape[1]}')
feat.head(3)